# Turb-DETR — Underwater Plastic Detection (Colab Training)

End-to-end training pipeline using **RT-DETR** via Ultralytics on underwater trash datasets.

| Item | Detail |
|---|---|
| Model | RT-DETR-L (COCO pretrained) |
| Datasets | Trash-ICRA19, UFO-120, TrashCan, RUIE |
| Framework | Ultralytics 8.3+ / PyTorch 2.2+ |
| Runtime | GPU (T4 or better) |

> **Before running:** Go to *Runtime → Change runtime type → **GPU*** 

## 1. Environment Setup

In [ ]:
# ── 1a. Check GPU availability ──────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# ── 1b. Clone repository & enter project directory ──────────
import os

REPO_URL = "https://github.com/csdeepak/turb-detr-underwater-detection.git"
REPO_DIR = "/content/turb-detr-underwater-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}")

%cd {REPO_DIR}

In [ ]:
# ── 1c. Install project dependencies ────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 1d. Verify installation & GPU ───────────────────────────
import torch
import ultralytics

ultralytics.checks()  # prints ultralytics + system info

print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Ultralytics : {ultralytics.__version__}")

## 2. Mount Google Drive & Link Datasets

Datasets are stored in a shared Google Drive folder:
`https://drive.google.com/drive/folders/1pFQ7s1dKNcCs8XoAqVko91mM_j2sHVSE`

Contains: **trash_icra19** · **ufo120** · **trashcan** · **ruie**

In [ ]:
# ── 2a. Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── 2b. Symlink datasets into project data/ folder ──────────
import os
from pathlib import Path

# Google Drive folder containing all datasets
DRIVE_DATASETS = Path("/content/drive/MyDrive/datasets")  # ← adjust if different

# Dataset names expected in that folder
DATASET_NAMES = ["trash_icra19", "ufo120", "trashcan", "ruie"]

# Create data/ directory and symlink each dataset
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

for name in DATASET_NAMES:
    src = DRIVE_DATASETS / name
    dst = data_dir / name
    if dst.exists() or dst.is_symlink():
        print(f"  ✓ {name:15s} → already linked")
        continue
    if src.exists():
        os.symlink(src, dst)
        print(f"  ✓ {name:15s} → linked from Drive")
    else:
        print(f"  ✗ {name:15s} → NOT FOUND at {src}")

print(f"\ndata/ contents: {[p.name for p in data_dir.iterdir()]}")

## 3. Verify Dataset Structure

In [ ]:
# ── 3a. Verify expected folder structure per dataset ─────────
from pathlib import Path

EXPECTED_SPLITS = ["train", "val", "test"]

for ds_name in ["trash_icra19", "ufo120", "trashcan", "ruie"]:
    ds_root = Path("data") / ds_name
    print(f"\n{'─'*50}")
    print(f"📂 {ds_name}  →  {'EXISTS' if ds_root.exists() else '⚠ MISSING'}")
    if not ds_root.exists():
        continue

    # Check for images/ and labels/ subdirectories per split
    for split in EXPECTED_SPLITS:
        img_dir = ds_root / "images" / split
        lbl_dir = ds_root / "labels" / split
        img_ok = img_dir.exists()
        lbl_ok = lbl_dir.exists()
        n_imgs = len(list(img_dir.glob("*"))) if img_ok else 0
        n_lbls = len(list(lbl_dir.glob("*"))) if lbl_ok else 0
        status = "✓" if (img_ok and lbl_ok) else "✗"
        print(f"  {status} {split:6s} — images: {n_imgs:>5,}  labels: {n_lbls:>5,}")

In [ ]:
# ── 3b. Print dataset statistics ─────────────────────────────
from pathlib import Path
from collections import Counter

def dataset_stats(ds_path: Path, class_names: list[str] | None = None):
    """Print image counts and class distribution for a YOLO-format dataset."""
    print(f"\n{'='*55}")
    print(f"  Dataset: {ds_path.name}")
    print(f"{'='*55}")

    total_imgs, total_objects = 0, 0
    class_counts = Counter()

    for split in ["train", "val", "test"]:
        img_dir = ds_path / "images" / split
        lbl_dir = ds_path / "labels" / split
        n_imgs = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        total_imgs += n_imgs

        if lbl_dir.exists():
            for lbl_file in lbl_dir.glob("*.txt"):
                for line in lbl_file.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if parts:
                        class_counts[int(parts[0])] += 1
                        total_objects += 1

        print(f"  {split:6s}: {n_imgs:>6,} images")

    print(f"  {'─'*40}")
    print(f"  Total : {total_imgs:>6,} images, {total_objects:>7,} objects")

    if class_counts:
        print(f"\n  Class distribution:")
        for cls_id in sorted(class_counts):
            name = class_names[cls_id] if class_names and cls_id < len(class_names) else f"class_{cls_id}"
            pct = class_counts[cls_id] / total_objects * 100
            bar = "█" * int(pct / 2)
            print(f"    {cls_id}: {name:15s} {class_counts[cls_id]:>6,}  ({pct:5.1f}%) {bar}")

# Run stats for Trash-ICRA19 (primary dataset)
trash_icra19_classes = ["plastic", "bottle", "can", "bag", "net"]
ds = Path("data/trash_icra19")
if ds.exists():
    dataset_stats(ds, trash_icra19_classes)
else:
    print("⚠ data/trash_icra19 not found — check symlinks above.")

## 4. Train Baseline RT-DETR Model

Using `configs/trash_icra19.yaml` as the data config.
Training via the Ultralytics API — all augmentations, scheduling, and logging are handled automatically.

In [ ]:
# ── 4a. Launch RT-DETR training ──────────────────────────────
from ultralytics import RTDETR

# Load COCO-pretrained RT-DETR-L
model = RTDETR("rtdetr-l.pt")

# Train on Trash-ICRA19
results = model.train(
    data="configs/trash_icra19.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=15,
    amp=True,
    workers=4,
    project="outputs",
    name="rtdetr_trash_icra19",
    save_period=10,
    exist_ok=True,
    verbose=True,
)

## 5. Evaluate & Visualize

In [ ]:
# ── 5a. Validate on test/val set ─────────────────────────────
metrics = model.val(data="configs/trash_icra19.yaml")

print(f"\nmAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

# ── 5b. Inference on sample images ───────────────────────────
from IPython.display import Image, display
from pathlib import Path
import glob

# Pick up to 4 test images for a quick visual check
test_dir = Path("data/trash_icra19/images/test")
sample_images = sorted(glob.glob(str(test_dir / "*")))[:4]

if sample_images:
    preds = model.predict(
        source=sample_images,
        imgsz=640,
        conf=0.25,
        save=True,
        project="outputs",
        name="rtdetr_predictions",
        exist_ok=True,
    )
    # Display saved prediction images
    pred_dir = Path("outputs/rtdetr_predictions")
    for img_path in sorted(pred_dir.glob("*"))[:4]:
        display(Image(filename=str(img_path), width=600))
else:
    print("⚠ No test images found — check dataset structure.")

In [ ]:
# ── 5c. Save best weights to Google Drive ────────────────────
import shutil
from pathlib import Path

src = Path("outputs/rtdetr_trash_icra19/weights/best.pt")
dst = Path("/content/drive/MyDrive/turb_detr_weights/")
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copy2(src, dst / "best.pt")
    # Also copy last.pt as backup
    last = src.parent / "last.pt"
    if last.exists():
        shutil.copy2(last, dst / "last.pt")
    print(f"✓ Weights saved to {dst}")
    print(f"  best.pt  : {src.stat().st_size / 1e6:.1f} MB")
else:
    print(f"✗ best.pt not found at {src}")
    print("  Check training output above for the correct path.")